In [1]:
import os
import re
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt


In [3]:

# =========================
# User settings
# =========================
DATA_DIR = "./adatas/RR/"   # 改成你的h5ad所在目录
OUT_DIR  = "./DE_pre_nres_vs_pre_res"
os.makedirs(OUT_DIR, exist_ok=True)

# 差异阈值（你可改）
PADJ_CUTOFF = 0.05
MIN_PCT_EXP = 0.10     # 过滤：基因在任一组表达细胞比例 >= 10%（避免极稀有假阳性）
METHOD = "wilcoxon"    # scanpy rank_genes_groups method

# 你要的clusters（按你的文字：LSC, MPP1, GMP1, EMP(Megakaryocyte)）
# 这些“key”只是输出用的名字，真正匹配靠文件名里 cluster token
TARGET_CLUSTERS = {
    "LSC": "HSCLSC",
    "MPP1": "MPPCLP1",
    "EMP_Megakaryocyte": "Megakaryocyte",
    "GMP1": "GMP1",          # 注意：目前你缺 pre_res_GMP1 文件，会自动跳过
}

# =========================
# Helpers
# =========================
def find_file(group_prefix: str, cluster_token: str, files: list[str]) -> str | None:
    """
    group_prefix: 'RR_pre_nres_' or 'RR_pre_res_'
    cluster_token: e.g. 'HSCLSC'
    """
    pat = re.compile(rf"^{re.escape(group_prefix)}{re.escape(cluster_token)}\.h5ad$")
    for f in files:
        if pat.match(f):
            return os.path.join(DATA_DIR, f)
    return None

def prep_adata(adata: sc.AnnData) -> sc.AnnData:
    """
    标准化流程：normalize_total + log1p
    如果你的h5ad里 X 已经是 lognorm，可以把这里改成“只做copy不处理”
    """
    ad = adata.copy()
    sc.pp.normalize_total(ad, target_sum=1e4)
    sc.pp.log1p(ad)
    # 不做HVG筛选：我们要全基因的上调列表
    return ad

def pct_expressed(X):
    # X: cells x genes (dense or sparse)
    if not hasattr(X, "toarray"):
        return np.mean(X > 0, axis=0)
    else:
        return np.mean((X > 0).toarray(), axis=0)

def de_upregulated(pre_nres_path: str, pre_res_path: str, cluster_name: str) -> tuple[pd.DataFrame, sc.AnnData]:
    ad_nres = prep_adata(sc.read_h5ad(pre_nres_path))
    ad_res  = prep_adata(sc.read_h5ad(pre_res_path))

    # 合并
    ad = ad_nres.concatenate(ad_res, batch_key="group", batch_categories=["pre_nres", "pre_res"])
    ad.obs["group"] = ad.obs["group"].astype(str)

    # DE
    sc.tl.rank_genes_groups(ad, groupby="group", groups=["pre_nres"], reference="pre_res", method=METHOD)

    # 取结果
    rg = ad.uns["rank_genes_groups"]
    df = pd.DataFrame({
        "gene": rg["names"]["pre_nres"],
        "log2fc_scanpy": rg["logfoldchanges"]["pre_nres"],
        "pval": rg["pvals"]["pre_nres"],
        "padj": rg["pvals_adj"]["pre_nres"],
        "score": rg["scores"]["pre_nres"],
    })

    # 计算表达比例 + 平均表达（log空间）
    genes = df["gene"].values
    # gene index
    gene_to_idx = {g:i for i,g in enumerate(ad.var_names)}
    idx = np.array([gene_to_idx[g] for g in genes if g in gene_to_idx], dtype=int)
    df = df[df["gene"].isin(ad.var_names)].reset_index(drop=True)

    X = ad.X
    grp = ad.obs["group"].values
    mask_nres = (grp == "pre_nres")
    mask_res  = (grp == "pre_res")

    X_nres = X[mask_nres, :][:, idx]
    X_res  = X[mask_res,  :][:, idx]

    df["pct_exp_pre_nres"] = pct_expressed(X_nres)
    df["pct_exp_pre_res"]  = pct_expressed(X_res)

    # mean expression (log1p normalized)
    def mean_expr(M):
        if not hasattr(M, "mean"):
            return np.mean(M, axis=0)
        m = M.mean(axis=0)
        return np.array(m).ravel()

    df["mean_log_pre_nres"] = mean_expr(X_nres)
    df["mean_log_pre_res"]  = mean_expr(X_res)

    # fold-change（在 log1p 空间差值不是严格FC；这里用 2**log2fc_scanpy 作为“近似FC(Scanpy)”）
    df["FC_approx"] = np.power(2.0, df["log2fc_scanpy"].astype(float))

    # 过滤：上调 + 显著 + 表达比例
    df_up = df[
        (df["padj"] < PADJ_CUTOFF) &
        (df["log2fc_scanpy"] > 0) &
        (
            (df["pct_exp_pre_nres"] >= MIN_PCT_EXP) |
            (df["pct_exp_pre_res"]  >= MIN_PCT_EXP)
        )
    ].copy()

    df_up.insert(0, "cluster", cluster_name)
    df_up = df_up.sort_values(["padj", "log2fc_scanpy"], ascending=[True, False]).reset_index(drop=True)

    return df_up, ad

def make_global_heatmap(genes_focus: list[str], adata_by_cluster: dict, out_png: str):
    """
    genes_focus: LSC里 FC>1.3 的基因
    adata_by_cluster: {cluster_name: concatenated AnnData with obs['group'] in {'pre_nres','pre_res'}}
    画一个 heatmap：行=genes，列=cluster_group（如 LSC_pre_nres, LSC_pre_res, ...）
    """
    # 构建矩阵：每个cluster的pre_nres/pre_res平均表达
    cols = []
    mats = []
    for cl, ad in adata_by_cluster.items():
        for grp in ["pre_nres", "pre_res"]:
            mask = (ad.obs["group"].values == grp)
            cols.append(f"{cl}_{grp}")
            # mean log expr
            vec = []
            for g in genes_focus:
                if g in ad.var_names:
                    j = np.where(ad.var_names == g)[0][0]
                    x = ad.X[mask, j]
                    if hasattr(x, "mean"):
                        m = x.mean()
                    else:
                        m = np.mean(x)
                    vec.append(float(np.array(m).ravel()[0]) if hasattr(m, "shape") else float(m))
                else:
                    vec.append(np.nan)
            mats.append(vec)

    M = np.array(mats).T  # genes x cols

    plt.figure(figsize=(max(8, len(cols)*0.8), max(6, len(genes_focus)*0.25)))
    plt.imshow(M, aspect="auto")
    plt.colorbar(label="Mean log1p(normalized) expression")
    plt.yticks(np.arange(len(genes_focus)), genes_focus, fontsize=7)
    plt.xticks(np.arange(len(cols)), cols, rotation=45, ha="right", fontsize=8)
    plt.title("Focused genes (LSC FC>1.3 in pre_nres vs pre_res) across clusters")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

# =========================
# Main
# =========================
all_files = [f for f in os.listdir(DATA_DIR) if f.endswith(".h5ad")]

results = []
adata_by_cluster = {}

for out_cluster_name, token in TARGET_CLUSTERS.items():
    pre_nres = find_file("RR_pre_nres_", token, all_files)
    pre_res  = find_file("RR_pre_res_",  token, all_files)

    if (pre_nres is None) or (pre_res is None):
        print(f"[SKIP] {out_cluster_name}: missing pair. pre_nres={pre_nres}, pre_res={pre_res}")
        continue

    print(f"[RUN] {out_cluster_name}: {os.path.basename(pre_nres)} vs {os.path.basename(pre_res)}")
    df_up, ad = de_upregulated(pre_nres, pre_res, out_cluster_name)

    out_csv = os.path.join(OUT_DIR, f"UP_pre_nres_vs_pre_res__{out_cluster_name}.csv")
    df_up.to_csv(out_csv, index=False)
    print(f"  -> saved: {out_csv} (n_up={len(df_up)})")

    results.append(df_up)
    adata_by_cluster[out_cluster_name] = ad

# 汇总表
if len(results) > 0:
    df_all = pd.concat(results, ignore_index=True)
    df_all.to_csv(os.path.join(OUT_DIR, "UP_pre_nres_vs_pre_res__ALL_CLUSTERS.csv"), index=False)

    # 提取 LSC 中 FC>1.3 基因做总图
    if "LSC" in df_all["cluster"].unique():
        lsc_focus = df_all[(df_all["cluster"] == "LSC") & (df_all["FC_approx"] > 1.3)]["gene"].drop_duplicates().tolist()
        pd.DataFrame({"gene": lsc_focus}).to_csv(os.path.join(OUT_DIR, "LSC_genes_FCgt1p3.csv"), index=False)
        print(f"[INFO] LSC FC>1.3 genes: {len(lsc_focus)}")

        if len(lsc_focus) > 0 and len(adata_by_cluster) > 0:
            out_png = os.path.join(OUT_DIR, "Heatmap_LSC_FCgt1p3_genes_across_clusters.png")
            make_global_heatmap(lsc_focus, adata_by_cluster, out_png)
            print(f"[PLOT] saved: {out_png}")
else:
    print("[DONE] No clusters were analyzed (no valid pre_nres/pre_res pairs found).")

print("[DONE]")


[RUN] LSC: RR_pre_nres_HSCLSC.h5ad vs RR_pre_res_HSCLSC.h5ad


C:\Users\abdul\AppData\Local\Temp\ipykernel_7924\2276534864.py:59: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  ad = ad_nres.concatenate(ad_res, batch_key="group", batch_categories=["pre_nres", "pre_res"])


  -> saved: ./DE_pre_nres_vs_pre_res\UP_pre_nres_vs_pre_res__LSC.csv (n_up=1142)
[RUN] MPP1: RR_pre_nres_MPPCLP1.h5ad vs RR_pre_res_MPPCLP1.h5ad


C:\Users\abdul\AppData\Local\Temp\ipykernel_7924\2276534864.py:59: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  ad = ad_nres.concatenate(ad_res, batch_key="group", batch_categories=["pre_nres", "pre_res"])


  -> saved: ./DE_pre_nres_vs_pre_res\UP_pre_nres_vs_pre_res__MPP1.csv (n_up=754)
[RUN] EMP_Megakaryocyte: RR_pre_nres_Megakaryocyte.h5ad vs RR_pre_res_Megakaryocyte.h5ad


C:\Users\abdul\AppData\Local\Temp\ipykernel_7924\2276534864.py:59: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  ad = ad_nres.concatenate(ad_res, batch_key="group", batch_categories=["pre_nres", "pre_res"])


  -> saved: ./DE_pre_nres_vs_pre_res\UP_pre_nres_vs_pre_res__EMP_Megakaryocyte.csv (n_up=895)
[SKIP] GMP1: missing pair. pre_nres=None, pre_res=./adatas/RR/RR_pre_res_GMP1.h5ad
[INFO] LSC FC>1.3 genes: 1138
[PLOT] saved: ./DE_pre_nres_vs_pre_res\Heatmap_LSC_FCgt1p3_genes_across_clusters.png
[DONE]
